### **Notebook 5 (etapa 5): SHAP aplicado a los modelos entrenados excluyendo los años de pandemia (2020 y 2021)**

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y el manejo de matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM durante la carga iterativa
import joblib  # Serialización y carga de modelos de Machine Learning (por consistencia del entorno)
import xgboost as xgb  # Algoritmo de ensamble avanzado (Gradient Boosting) para entrenamiento predictivo multiclase
from sklearn.ensemble import RandomForestClassifier  # Algoritmo de ensamble basado en múltiples árboles de decisión
from sklearn.model_selection import train_test_split  # Función para dividir los datos en conjuntos de entrenamiento y prueba
import shap  # Biblioteca principal basada en teoría de juegos para la explicabilidad de los modelos
from sklearn.metrics import f1_score  # Métrica principal para evaluar el rendimiento predictivo del modelo

# 1. Configuración de Rutas y Variables
# Directorio donde se ubican los datos anuales procesados y estandarizados
dir_entrada = "../../Datos/Datos procesados"
# Directorio destino exclusivo para la Fase 5: Análisis de Sensibilidad (Aislamiento de la Pandemia)
dir_resultados = "../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia"
# Crear el directorio de resultados si aún no existe en el disco duro
os.makedirs(dir_resultados, exist_ok=True)

# Lista exhaustiva de todas las columnas que se extraerán de los archivos CSV
columnas_a_cargar = [
    'CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER',
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 
    'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS'
]

# Lista de variables categóricas que deberán ser transformadas a variables dummy (One-Hot Encoding)
vars_para_ohe = [
    'COMORBILIDAD_PRINCIPAL', 'ES_EXTRANJERO', 'ES_PUEBLO_ORIGINARIO', 
    'TIPO_PROCEDIMIENTO', 'ESPECIALIDAD_MEDICA', 'REGION', 'SERVICIOINGRESO', 
    'SEXO', 'TIPO_DIAGNOSTICO_ONCO', 'TIPO_INGRESO', 'TIPO_PREVISION', 
    'TIPO_PROCEDENCIA', 'CATEGORIA_CANCER' 
]

# AÑOS A INCLUIR (EXCLUYENDO 2020 y 2021)
# Se eliminan los años fuertemente afectados por las cuarentenas y el colapso hospitalario de COVID-19
años_sin_pandemia = [2019, 2022, 2023, 2024]

# Imprimir encabezado del análisis en consola
print("="*70)
print("INICIANDO ANÁLISIS DE SENSIBILIDAD (SIN PANDEMIA 2020-2021)")
print("="*70)

# 2. Cargar y Procesar Datos
# Inicializar lista temporal para acumular los datos anuales
df_lista = []
for año in años_sin_pandemia:
    # Construir ruta dinámica para el archivo correspondiente al año iterado
    ruta = os.path.join(dir_entrada, f"GRD_PROCESADO_{año}_DERIVADAS.csv")
    if os.path.exists(ruta):
        print(f"Cargando {año}...")
        # Leer archivo extrayendo solo las columnas necesarias para no desbordar la memoria RAM
        df_temp = pd.read_csv(ruta, usecols=columnas_a_cargar, low_memory=False)
        # Limpiar la categoría de cáncer extrayendo solo el prefijo principal (previo a los dos puntos)
        df_temp['CATEGORIA_CANCER'] = df_temp['CATEGORIA_CANCER'].astype(str).str.split(':').str[0].str.replace('-', '_').str.strip()
        # Apilar el dataframe anual en la lista
        df_lista.append(df_temp)

# Unificar todos los dataframes anuales en un único conjunto de datos global
df_crudo = pd.concat(df_lista, ignore_index=True)
# Destruir la lista temporal y forzar la recolección de basura
del df_lista; gc.collect()

print("Aplicando One-Hot Encoding...")
# Aplicar transformación de variables categóricas a numéricas binarias (OHE), eliminando la categoría base
df_ohe = pd.get_dummies(df_crudo, columns=vars_para_ohe, drop_first=True)
# Normalizar los nombres de las columnas para evitar errores de sintaxis (mayúsculas, guiones bajos)
df_ohe.columns = df_ohe.columns.str.replace(' ', '_').str.replace('-', '_').str.upper()
# Eliminar el dataframe crudo de la memoria
del df_crudo; gc.collect()

# Optimización extrema de memoria reduciendo la precisión de los tipos de datos (Downcasting)
for col in df_ohe.select_dtypes(include=['float64']).columns:
    df_ohe[col] = df_ohe[col].astype('float32')  # Reducir decimales de 64 a 32 bits
for col in df_ohe.select_dtypes(include=['int64']).columns:
    df_ohe[col] = df_ohe[col].astype('int32')    # Reducir enteros de 64 a 32 bits

print("Separando cohorte oncológica y control...")
# Identificar todas las columnas OHE derivadas que corresponden a la presencia de algún tipo de cáncer
columnas_cancer = [col for col in df_ohe.columns if col.startswith('CATEGORIA_CANCER_') and 'SIN_CANCER' not in col]
# Crear una máscara booleana: Pacientes que tienen un 1 en al menos una de las columnas de cáncer
mask_onco = df_ohe[columnas_cancer].sum(axis=1) > 0

# Separar físicamente la cohorte de pacientes oncológicos y los pacientes de control
df_onco = df_ohe[mask_onco]
df_control = df_ohe[~mask_onco]
# Liberar el dataset OHE global
del df_ohe; gc.collect()

# 3. Función de Entrenamiento y Extracción SHAP
def entrenar_y_extraer_shap(target_name, es_rf=False):
    """
    Descripción:
        Entrena un nuevo modelo (Random Forest o XGBoost) utilizando un dataset que excluye 
        completamente los años de la pandemia de COVID-19 (2020 y 2021). Posteriormente, 
        extrae los valores SHAP sobre una muestra de prueba y los convierte a porcentajes. 
        Este análisis busca verificar si la importancia de las variables clínicas se mantiene 
        estable cuando se aísla el shock estadístico de la pandemia.

    Entradas:
        - target_name (str): Nombre de la variable objetivo a evaluar ('MORTALIDAD', 'SEVERIDAD' o 'CONSUMO_RECURSOS').
        - es_rf (bool): Bandera que determina el algoritmo a usar: True para Random Forest (Mortalidad binaria), 
                        False para XGBoost (Problemas multiclase).

    Salidas:
        - None: La función no retorna variables, pero calcula, procesa y guarda directamente en 
          el disco local un archivo CSV con las importancias porcentuales sin el sesgo pandémico.
    """
    print(f"\n--- Modelando {target_name} (Sin Pandemia) ---")
    
    # Dividir la cohorte oncológica en conjunto de entrenamiento (80%) y validación (20%) manteniendo la proporción de clases
    df_onco_train, df_onco_test = train_test_split(df_onco, test_size=0.20, random_state=42, stratify=df_onco[target_name])
    
    # Equilibrar el entrenamiento extrayendo una muestra de pacientes de control del mismo tamaño que la oncológica
    n_onco = len(df_onco_train)
    df_control_train = df_control.sample(n=n_onco, random_state=42)
    
    # Consolidar el dataframe de entrenamiento definitivo
    df_train = pd.concat([df_onco_train, df_control_train], ignore_index=True)
    
    # Lista de variables objetivo que no deben usarse como predictores
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD']
    # Configurar matrices independientes (X) y vector dependiente (y)
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    # Instanciar el modelo con los hiperparámetros óptimos identificados en las etapas anteriores
    if es_rf:
        # Configuración para clasificación Binaria (Mortalidad)
        modelo = RandomForestClassifier(n_estimators=500, max_depth=35, min_samples_split=10, class_weight='balanced', n_jobs=-1, random_state=42)
    else:
        # Configuración para clasificación Multiclase (Severidad y Consumo)
        modelo = xgb.XGBClassifier(learning_rate=0.3, max_depth=10, tree_method='hist', n_jobs=-1, random_state=42)
        
    print("Entrenando modelo...")
    # Ejecutar el ajuste del modelo con la data libre de pandemia
    modelo.fit(X_train, y_train)
    
    # Preparar el set de prueba oncológico, asegurando la misma alineación de columnas que el set de entrenamiento
    X_test_onco = df_onco_test.drop(columns=cols_drop, errors='ignore').reindex(columns=X_train.columns, fill_value=0)
    # Inferir predicciones para evaluar la integridad del nuevo modelo
    y_pred = modelo.predict(X_test_onco)
    
    # Inicializar el motor de explicabilidad SHAP para árboles
    explainer = shap.TreeExplainer(modelo)
    
    # Extraer una muestra aleatoria de máximo 5000 pacientes para no saturar la RAM durante el cálculo SHAP
    X_sample = X_test_onco.sample(n=min(5000, len(X_test_onco)), random_state=42)
    # Imprimir métrica rápida de rendimiento (F1-binary para RF, F1-macro para XGB)
    print(f"F1-Score en prueba: {f1_score(df_onco_test[target_name], y_pred, average='macro' if not es_rf else 'binary'):.4f}")
    
    print("Calculando SHAP y normalizando a porcentajes...")
    
    # --- FLUJO SHAP PARA RANDOM FOREST (MORTALIDAD / BINARIO) ---
    if es_rf: 
        # Extraer valores usando aproximación para acelerar el cómputo
        shap_values = explainer.shap_values(X_sample, check_additivity=False, approximate=True)
        # Adaptar extracción dependiendo de si SHAP retorna una lista (2 clases) o matriz directa
        shap_mat = shap_values[1] if isinstance(shap_values, list) else np.array(shap_values)[:, :, 1]
        
        # Filtro de variables que quedaron constantes (sin varianza) en la muestra extraída
        varianzas = X_sample.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        # Forzar la inclusión de 'SIN_CANCER' en el filtro si por algún error sobrevivió
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_sample.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        # Purgar columnas constantes de la muestra y de la matriz de resultados SHAP
        if cols_a_eliminar:
            idx_a_eliminar = [X_sample.columns.get_loc(col) for col in cols_a_eliminar]
            X_sample = X_sample.drop(columns=cols_a_eliminar)
            shap_mat = np.delete(shap_mat, idx_a_eliminar, axis=1)

        # Cálculo de porcentajes para modelo Binario 
        # (La magnitud absoluta de la clase 1 y el total son matemáticamente equivalentes en enfoques binarios)
        shap_abs = np.abs(shap_mat).mean(axis=0)
        shap_pct_total = (shap_abs / shap_abs.sum()) * 100
        
        # Consolidar los resultados en un DataFrame exportable
        df_imp = pd.DataFrame({
            'Variable': X_sample.columns, 
            'Impacto_Total_SinPandemia': shap_pct_total,
            'Impacto_Clase1_SinPandemia': shap_pct_total 
        })

    # --- FLUJO SHAP PARA XGBOOST (SEVERIDAD Y CONSUMO / MULTICLASE) ---
    else: 
        # Extraer tensor SHAP 3D directo del motor optimizado para XGBoost
        shap_values = explainer(X_sample).values
        
        # Filtro de variables constantes (sin varianza) dentro de la muestra extraída
        varianzas = X_sample.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_sample.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        # Purgar dimensiones sin varianza del dataset y del tensor 3D de SHAP
        if cols_a_eliminar:
            idx_a_eliminar = [X_sample.columns.get_loc(col) for col in cols_a_eliminar]
            X_sample = X_sample.drop(columns=cols_a_eliminar)
            shap_values = np.delete(shap_values, idx_a_eliminar, axis=1)

        # Promediar el impacto absoluto por feature cruzando los pacientes
        shap_abs_classes = np.abs(shap_values).mean(axis=0) 
        
        # 1. Porcentaje Total (Se suman las contribuciones de todas las clases para crear un ranking global)
        impacto_crudo_total = shap_abs_classes.sum(axis=1)
        shap_pct_total = (impacto_crudo_total / impacto_crudo_total.sum()) * 100
        
        # Iniciar la consolidación del DataFrame exportable con la métrica unificada
        df_imp = pd.DataFrame({
            'Variable': X_sample.columns, 
            'Impacto_Total_SinPandemia': shap_pct_total
        })
        
        # 2. Porcentaje Específico extraído directamente de la Clase de Alto Riesgo
        if target_name == 'SEVERIDAD': 
            # Clase 3 representa la Severidad Mayor
            impacto_crudo_c3 = shap_abs_classes[:, 3]
            df_imp['Impacto_Clase3_SinPandemia'] = (impacto_crudo_c3 / impacto_crudo_c3.sum()) * 100
        elif target_name == 'CONSUMO_RECURSOS': 
            # Clase 2 representa el Consumo Alto
            impacto_crudo_c2 = shap_abs_classes[:, 2]
            df_imp['Impacto_Clase2_SinPandemia'] = (impacto_crudo_c2 / impacto_crudo_c2.sum()) * 100

    # Ordenar la tabla de resultados de mayor a menor relevancia global
    df_imp = df_imp.sort_values(by='Impacto_Total_SinPandemia', ascending=False)
    
    # Exportar el ranking final en formato CSV dentro de la carpeta de Sensibilidad
    ruta_csv = os.path.join(dir_resultados, f"SHAP_Porcentajes_SinPandemia_{target_name}.csv")
    df_imp.to_csv(ruta_csv, index=False)
    print(f"-> Archivo con porcentajes guardado en {ruta_csv}")
    
    # Destruir variables pesadas asociadas a la iteración actual para liberar recursos
    del modelo, X_train, y_train, df_train; gc.collect()

# ====================================================================
# 4. Ejecución del pipeline de sensibilidad iterando sobre cada objetivo
# ====================================================================

# Predecir Mortalidad utilizando el algoritmo Binario (Random Forest)
entrenar_y_extraer_shap('MORTALIDAD', es_rf=True)

# Predecir Severidad utilizando el algoritmo Multiclase (XGBoost)
entrenar_y_extraer_shap('SEVERIDAD', es_rf=False)

# Predecir Consumo de Recursos utilizando el algoritmo Multiclase (XGBoost)
entrenar_y_extraer_shap('CONSUMO_RECURSOS', es_rf=False)

# Mensaje final de confirmación
print("\n=== PROCESO FINALIZADO ===")

INICIANDO ANÁLISIS DE SENSIBILIDAD (SIN PANDEMIA 2020-2021)
Cargando 2019...
Cargando 2022...
Cargando 2023...
Cargando 2024...
Aplicando One-Hot Encoding...
Separando cohorte oncológica y control...

--- Modelando MORTALIDAD (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.4205
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia/SHAP_Porcentajes_SinPandemia_MORTALIDAD.csv

--- Modelando SEVERIDAD (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.7751
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia/SHAP_Porcentajes_SinPandemia_SEVERIDAD.csv

--- Modelando CONSUMO_RECURSOS (Sin Pandemia) ---
Entrenando modelo...
F1-Score en prueba: 0.7592
Calculando SHAP y normalizando a porcentajes...
-> Archivo con porcentajes guardado en ../../Resultados/Resultados (etapa 5)/Sensibilid

In [ ]:
"""
Descripción:
    Script de evaluación cruzada para el Análisis de Sensibilidad (Aislamiento de Pandemia).
    Compara los resultados de explicabilidad SHAP del modelo original (que incluye los años 
    de pandemia) versus el modelo entrenado excluyendo 2020 y 2021. 
    Calcula dinámicamente las variaciones en el ranking de importancia y las diferencias 
    porcentuales (Deltas) de las 20 variables más críticas para cada variable objetivo.

Entradas:
    - Archivos CSV con los resultados SHAP originales (ubicados en las carpetas de Etapa 5).
    - Archivos CSV con los resultados SHAP sin pandemia (generados en el paso anterior).

Salidas:
    - Archivos CSV consolidados con la comparativa del Top 20 de variables, detallando 
      sus rankings en ambos escenarios y la magnitud del cambio (Delta) general y por clase.
"""

import pandas as pd  # Permite el manejo, cruce y análisis de estructuras de datos tabulares (DataFrames)
import os  # Interacción con el sistema operativo (creación de directorios y manejo de rutas locales)

# 1. Configuración de rutas
# Rutas hacia los reportes SHAP originales (con toda la data, incluyendo años COVID-19)
dir_orig_mort = "../../Resultados/Resultados (etapa 5)/SHAP_MORTALIDAD/Valores SHAP (oncologicos)"
dir_orig_sev = "../../Resultados/Resultados (etapa 5)/SHAP_SEVERIDAD/Valores SHAP (oncologicos)"
dir_orig_cons = "../../Resultados/Resultados (etapa 5)/SHAP_CONSUMO_RECURSOS/Valores SHAP (oncologicos)"

# Ruta hacia los reportes SHAP generados al excluir 2020 y 2021
dir_sin_pandemia = "../../Resultados/Resultados (etapa 5)/Sensibilidad_Pandemia"

# Imprimir encabezado de la ejecución en consola
print("="*60)
print("CALCULANDO DIFERENCIAS (SIN PANDEMIA - ORIGINAL)")
print("="*60)

# ====================================================================
# --- 1. MORTALIDAD ---
# ====================================================================
print("\nProcesando Mortalidad...")
# Cargar el CSV original con los porcentajes de impacto SHAP para Mortalidad
df_mort_orig = pd.read_csv(os.path.join(dir_orig_mort, "SHAP_Valores_MORTALIDAD_ONCO_PORCENTAJES.csv"))
# Cargar el CSV sin pandemia con los porcentajes de impacto SHAP para Mortalidad
df_mort_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_MORTALIDAD.csv"))

# Calcular rankings REALES ordenando los impactos de mayor a menor antes de unir las tablas
# El método 'first' asegura que no haya empates (asigna posiciones secuenciales)
df_mort_orig['Ranking_Original'] = df_mort_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_mort_sp['Ranking_Sin_Pandemia'] = df_mort_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Unir ambos dataframes utilizando el nombre de la 'Variable' como llave (Inner Join)
df_mort_merge = pd.merge(df_mort_orig, df_mort_sp, on='Variable', how='inner')

# Calcular la diferencia (Delta) de impacto porcentual total (Nuevo escenario - Original)
df_mort_merge['Delta_Impacto_Total'] = df_mort_merge['Impacto_Total_SinPandemia'] - df_mort_merge['Impacto_Total']
# Calcular la diferencia (Delta) de impacto porcentual específico para la Clase 1 (Fallecido)
df_mort_merge['Delta_Clase1'] = df_mort_merge['Impacto_Clase1_SinPandemia'] - df_mort_merge['Clase_1']

# Filtrar para retener únicamente las variables que pertenecían al Top 20 en el modelo original y ordenarlas
df_mort_top20 = df_mort_merge[df_mort_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()

# Seleccionar solo las columnas de interés para un reporte limpio
cols_mort = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_1', 'Impacto_Clase1_SinPandemia', 'Delta_Clase1']
# Exportar la tabla de diferencias al directorio de Sensibilidad
df_mort_top20[cols_mort].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_MORTALIDAD.csv"), index=False)
print("-> Diferencias_Pandemia_MORTALIDAD.csv guardado.")


# ====================================================================
# --- 2. SEVERIDAD ---
# ====================================================================
print("\nProcesando Severidad...")
# Cargar CSV original porcentual para Severidad
df_sev_orig = pd.read_csv(os.path.join(dir_orig_sev, "SHAP_Valores_SEVERIDAD_ONCO_PORCENTAJES.csv"))
# Cargar CSV sin pandemia porcentual para Severidad
df_sev_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_SEVERIDAD.csv"))

# Calcular jerarquías (rankings) en base al peso de cada variable en sus respectivos escenarios
df_sev_orig['Ranking_Original'] = df_sev_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_sev_sp['Ranking_Sin_Pandemia'] = df_sev_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Cruzar datos por variable
df_sev_merge = pd.merge(df_sev_orig, df_sev_sp, on='Variable', how='inner')

# Calcular deltas generales
df_sev_merge['Delta_Impacto_Total'] = df_sev_merge['Impacto_Total_SinPandemia'] - df_sev_merge['Impacto_Total']
# Calcular deltas específicos para la Clase 3 (Severidad Mayor / Mayor Riesgo)
df_sev_merge['Delta_Clase3'] = df_sev_merge['Impacto_Clase3_SinPandemia'] - df_sev_merge['Clase_3']

# Recortar el dataset a las 20 características líderes originales
df_sev_top20 = df_sev_merge[df_sev_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()

# Limpiar y estructurar las columnas de salida
cols_sev = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_3', 'Impacto_Clase3_SinPandemia', 'Delta_Clase3']
# Exportar reporte de Severidad
df_sev_top20[cols_sev].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_SEVERIDAD.csv"), index=False)
print("-> Diferencias_Pandemia_SEVERIDAD.csv guardado.")


# ====================================================================
# --- 3. CONSUMO DE RECURSOS ---
# ====================================================================
print("\nProcesando Consumo de Recursos...")
# Cargar CSV original porcentual para Consumo de Recursos
df_cons_orig = pd.read_csv(os.path.join(dir_orig_cons, "SHAP_Valores_CONSUMO_RECURSOS_ONCO_PORCENTAJES.csv"))
# Cargar CSV sin pandemia porcentual para Consumo de Recursos
df_cons_sp = pd.read_csv(os.path.join(dir_sin_pandemia, "SHAP_Porcentajes_SinPandemia_CONSUMO_RECURSOS.csv"))

# Establecer la posición ordinal (Ranking) de los predictores en ambos datasets
df_cons_orig['Ranking_Original'] = df_cons_orig['Impacto_Total'].rank(ascending=False, method='first').astype(int)
df_cons_sp['Ranking_Sin_Pandemia'] = df_cons_sp['Impacto_Total_SinPandemia'].rank(ascending=False, method='first').astype(int)

# Emparejar la información
df_cons_merge = pd.merge(df_cons_orig, df_cons_sp, on='Variable', how='inner')

# Extraer el cambio métrico general
df_cons_merge['Delta_Impacto_Total'] = df_cons_merge['Impacto_Total_SinPandemia'] - df_cons_merge['Impacto_Total']
# Extraer el cambio métrico focalizado en la Clase 2 (Consumo Alto)
df_cons_merge['Delta_Clase2'] = df_cons_merge['Impacto_Clase2_SinPandemia'] - df_cons_merge['Clase_2']

# Filtrar preservando el marco de referencia del Top 20 original
df_cons_top20 = df_cons_merge[df_cons_merge['Ranking_Original'] <= 20].sort_values(by='Ranking_Original').copy()

# Preparar las columnas requeridas para el entregable
cols_cons = ['Ranking_Original', 'Ranking_Sin_Pandemia', 'Variable', 'Impacto_Total', 'Impacto_Total_SinPandemia', 'Delta_Impacto_Total', 'Clase_2', 'Impacto_Clase2_SinPandemia', 'Delta_Clase2']
# Exportar reporte de Consumo de Recursos
df_cons_top20[cols_cons].to_csv(os.path.join(dir_sin_pandemia, "Diferencias_Pandemia_CONSUMO_RECURSOS.csv"), index=False)
print("-> Diferencias_Pandemia_CONSUMO_RECURSOS.csv guardado.")

# Mensaje de término del análisis
print("\n=== CÁLCULO DE DIFERENCIAS FINALIZADO ===")

CALCULANDO DIFERENCIAS (SIN PANDEMIA - ORIGINAL)

Procesando Mortalidad...
-> Diferencias_Pandemia_MORTALIDAD.csv guardado.

Procesando Severidad...
-> Diferencias_Pandemia_SEVERIDAD.csv guardado.

Procesando Consumo de Recursos...
-> Diferencias_Pandemia_CONSUMO_RECURSOS.csv guardado.

=== CÁLCULO DE DIFERENCIAS FINALIZADO ===
